# 🎙️ Tetrax STT — Whisper Small Multilingual Fine-Tuning & CPaaS Deployment
Fine-tuning OpenAI Whisper Small on Google FLEURS (`hi_in`, `mr_in`, `en_us`) using GPU acceleration and integrating STT API into Tetrax CPaaS.

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Install Dependencies & Setup Environment

In [ ]:
!pip install -q transformers datasets[audio] accelerate evaluate jiwer soundfile librosa pandas pyyaml fastapi uvicorn python-multipart

### Step 3: Setup Repository Directory

In [ ]:
# If using Git repo:
# !git clone YOUR_REPO_URL tetrax-stt
# %cd tetrax-stt

# Or if zip uploaded to Colab:
# !unzip tetrax-stt.zip
# %cd tetrax-stt

### Step 4: Check GPU & Environment Setup

In [ ]:
!python scripts/check_environment.py

### Step 5: Download/Prepare Dataset

In [ ]:
# Download google/fleurs (hi_in, mr_in, en_us)
!python scripts/download_fleurs_data.py

In [ ]:
# Validate manifest files
!python scripts/prepare_dataset.py

### Step 6: Fine-Tune Whisper Small

In [ ]:
!python scripts/train.py --max_steps 500 --batch_size 4 --grad_accum 4 --learning_rate 1e-5

### Step 7: Evaluate Model Performance (WER)

In [ ]:
!python scripts/evaluate.py

In [ ]:
import pandas as pd
df = pd.read_csv('reports/wer_by_language.csv')
display(df)

### Step 8: Save Fine-Tuned Model to Google Drive

In [ ]:
!cp -r models/tetrax-stt-v1 /content/drive/MyDrive/tetrax-stt-v1
!zip -r tetrax-stt-v1.zip models/tetrax-stt-v1
print("Model checkpoint saved successfully to Google Drive and ready for download!")

### Step 9: Test Model Inference in Python

In [ ]:
import sys
sys.path.append('src')
from tetrax_stt.model import TetraxSTTModel

# Load fine-tuned model checkpoint
model = TetraxSTTModel('models/tetrax-stt-v1')

# Transcribe a sample audio file
sample_audio = 'data/audio/hi/test/hi_test_00000.wav'
result = model.transcribe(sample_audio, language='hi')

print("Transcription Result:", result['text'])
print("Detected Language:", result['language'])

### Step 10: How to Implement & Integrate STT API Service in Tetrax CPaaS

#### 1. Run the FastAPI Production Server
In your local environment or GPU cloud server, start the API service:
```bash
uvicorn tetrax_stt.api:app --app-dir src --host 0.0.0.0 --port 8000
```

#### 2. Node.js / Express CPaaS Backend Integration Example
In your `tetrax-backend` (Node.js/TypeScript), send audio recordings to the STT API:
```javascript
const fs = require('fs');
const FormData = require('form-data');
const axios = require('axios');

async function transcribeAudio(audioFilePath, language = 'hi') {
  const formData = new FormData();
  formData.append('file', fs.createReadStream(audioFilePath));
  formData.append('language', language);

  const response = await axios.post('http://localhost:8000/v1/audio/transcriptions', formData, {
    headers: formData.getHeaders()
  });

  return response.data; // { text: '...', language: 'hi', model: 'models/tetrax-stt-v1' }
}
```

#### 3. Python CPaaS Backend Integration Example
```python
import requests

def transcribe_speech(audio_bytes, language='hi'):
    files = {'file': ('sample.wav', audio_bytes, 'audio/wav')}
    data = {'language': language}
    response = requests.post('http://localhost:8000/v1/audio/transcriptions', files=files, data=data)
    return response.json()
```